### Link Grabber


In [6]:
from selenium import webdriver
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd
import time
import random

options = Options()
service = Service(ChromeDriverManager().install())


In [ ]:
def grab_tiktok_links(
        goal: int = 100,
        url: str = 'https://www.tiktok.com/tag/ai',
        a_class: str = 'css-1undbtb-7937d88b--AVideoContainer',
        wait_time: int = 1,
        ) -> list[str]:
    """
    Args:
        goal: Number of links seen before returning
        url: Tiktok hashtag url
        a_class: <a> container class for lookup
        wait_time: Seconds to pause between requesting more links

    Returns:
        links: Unique urls. Size not match goal.
    """
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(url)

    wait = WebDriverWait(driver, timeout=2)
    html = driver.find_element(By.TAG_NAME,"html")

    number_of_links = 0

    while number_of_links < goal:
        wait.until(
            lambda d: len(d.find_elements(By.CLASS_NAME, a_class)) != number_of_links
        )

        source = driver.page_source
        soup = BeautifulSoup(source, "html.parser")
        a = soup.find_all("a", {"class":a_class})
        number_of_links = len(a)

        time.sleep(random.gauss(1, 0.5))  

        html.send_keys(Keys.END)
    driver.quit()

    links = set()
    
    for vid in a:
        url = vid['href']
        links.add(url)
    
    return list(links)


In [8]:
def save_links(
        links: list,
        path: Path = Path("data"),
        filename: str = "urls.txt"
        ) -> None:
    df = pd.DataFrame(links, columns=["url"])

    path.mkdir(parents=True, exist_ok=True)
    df.to_csv(path / filename, index=False)

In [9]:
res = grab_tiktok_links(goal = 50)
save_links(res)